In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# Укажи путь к папке Raw Data
DATA_DIR = Path(r".\Датасет с ресторанами")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Папка не найдена: {DATA_DIR}")

print("Папка найдена:", DATA_DIR)

Папка найдена: Датасет с ресторанами


In [2]:
files = [f for f in DATA_DIR.iterdir() if f.is_file()]

print(f"Всего файлов: {len(files)}\n")

for file in files:
    print(file.name, f"({file.stat().st_size / 1024:.1f} KB)")

Всего файлов: 11

ingredients.csv (69.8 KB)
menuitem.csv (18313.1 KB)
menu_items.csv (708.9 KB)
portion_uom_types.csv (0.1 KB)
pos_ordersale.csv (23843.6 KB)
recipes.csv (483.8 KB)
recipe_ingredient_assignments.csv (149.2 KB)
recipe_sub_recipe_assignments.csv (129.6 KB)
store_restaurant.xlsx (9.1 KB)
sub_recipes.csv (8.2 KB)
sub_recipe_ingr_assignments.csv (3.2 KB)


In [5]:
tables = {}

for file in files:
    ext = file.suffix.lower()

    try:
        if ext == ".csv":
            try:
                df = pd.read_csv(file, sep=None, engine="python", encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(file, sep=None, engine="python", encoding="cp1252")

            tables[file.stem.lower()] = df

        elif ext in [".xlsx", ".xls"]:
            excel = pd.ExcelFile(file)

            for sheet in excel.sheet_names:
                df = pd.read_excel(file, sheet_name=sheet)
                name = f"{file.stem}_{sheet}".lower()
                tables[name] = df

        print(f"Загружен: {file.name}")

    except Exception as e:
        print(f"Ошибка при загрузке {file.name}: {e}")

print(f"\nВсего загружено таблиц: {len(tables)}")

Загружен: ingredients.csv
Загружен: menuitem.csv
Загружен: menu_items.csv
Загружен: portion_uom_types.csv
Загружен: pos_ordersale.csv
Загружен: recipes.csv
Загружен: recipe_ingredient_assignments.csv
Загружен: recipe_sub_recipe_assignments.csv
Загружен: store_restaurant.xlsx
Загружен: sub_recipes.csv
Загружен: sub_recipe_ingr_assignments.csv

Всего загружено таблиц: 11


In [6]:
for name, df in tables.items():
    print("=" * 80)
    print(f"ТАБЛИЦА: {name}")
    print(f"Количество строк: {df.shape[0]}")
    print(f"Количество столбцов: {df.shape[1]}")
    print("\nСтолбцы:")
    print(df.columns.tolist())

ТАБЛИЦА: ingredients
Количество строк: 673
Количество столбцов: 4

Столбцы:
['IngredientName', 'IngredientShortDescription', 'IngredientId', 'PortionUOMTypeId']
ТАБЛИЦА: menuitem
Количество строк: 91431
Количество столбцов: 15

Столбцы:
['MD5KEY_MENUITEM', 'MD5KEY_ORDERSALE', 'CategoryDescription', 'DepartmentDescription', 'Description', 'StoreNumber', 'TaxInclusiveAmount', 'TaxAmount', 'AdjustedPrice', 'DiscountAmount', 'Price', 'Quantity', 'PLU', 'Id', 'date']
ТАБЛИЦА: menu_items
Количество строк: 8005
Количество столбцов: 5

Столбцы:
['MenuItemName', 'MenuItemDescription', 'PLU', 'MenuItemId', 'RecipeId']
ТАБЛИЦА: portion_uom_types
Количество строк: 5
Количество столбцов: 2

Столбцы:
['PortionTypeDescription', 'PortionUOMTypeId']
ТАБЛИЦА: pos_ordersale
Количество строк: 43228
Количество столбцов: 9

Столбцы:
['MD5KEY_ORDERSALE', 'ChangeReceived', 'OrderNumber', 'TaxInclusiveAmount', 'TaxAmount', 'MealLocation', 'TransactionId', 'StoreNumber', 'date']
ТАБЛИЦА: recipes
Количество стро

In [7]:
missing_results = []

for name, df in tables.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        missing_results.append({
            "Таблица": name,
            "Столбец": column,
            "Всего записей": len(df),
            "Пропусков": missing_count,
            "Доля пропусков, %": round(
                missing_count / len(df) * 100, 2
            ) if len(df) > 0 else 0
        })

missing_df = pd.DataFrame(missing_results)

missing_df = missing_df[missing_df["Пропусков"] > 0]

missing_df.sort_values(
    by="Доля пропусков, %",
    ascending=False
)

,Таблица,Столбец,Всего записей,Пропусков,"Доля пропусков, %"
3,ingredients,PortionUOMTypeId,673,36,5.35


In [ ]:
duplicate_results = []

for name, df in tables.items():
    duplicate_count = df.duplicated().sum()

    duplicate_results.append({
        "Таблица": name,
        "Всего записей": len(df),
        "Полные дубликаты": duplicate_count,
        "Доля дубликатов, %": round(
            duplicate_count / len(df) * 100, 2
        ) if len(df) > 0 else 0
    })

duplicates_df = pd.DataFrame(duplicate_results)

duplicates_df

,Таблица,Всего записей,Полные дубликаты,"Доля дубликатов, %"
0,ingredients,673,0,0.0
1,menuitem,91431,0,0.0
2,menu_items,8005,0,0.0
3,portion_uom_types,5,0,0.0
4,pos_ordersale,43228,0,0.0
5,recipes,7458,0,0.0
6,recipe_ingredient_assignments,14308,0,0.0
7,recipe_sub_recipe_assignments,13890,0,0.0
8,store_restaurant_sheet1,4,0,0.0
9,sub_recipes,81,0,0.0


In [21]:
sales_candidates = [
    name for name in tables
    if "menuitem" in name.lower()
]

print("Найденные таблицы продаж:", sales_candidates)

Найденные таблицы продаж: ['menuitem']


In [22]:
sales = tables[sales_candidates[0]]

display(sales.head(10))

print("\nТипы данных:")
display(sales.dtypes)

print("\nОсновные статистические показатели:")
display(sales.describe(include="all").T)

,MD5KEY_MENUITEM,MD5KEY_ORDERSALE,CategoryDescription,DepartmentDescription,Description,StoreNumber,TaxInclusiveAmount,TaxAmount,AdjustedPrice,DiscountAmount,Price,Quantity,PLU,Id,date
0,001E0224E4EDA63D6FED96D59E0C1E76,072D3B76BE58E4553A373AF3206D2CA5,Footlong,Footlong Sub,Chickn Strips FtLong,46673,0,0.6975,7.75,0.00,7.75,1,111000041,326,15-03-05
1,003E4E20196B2F5BDD33CE06D2943EF4,14811D4F3458058007B0B6908C2DA393,Fountain,Fountain Drink 21 oz.,21oz Fountain Drink,46673,0,0.1530,1.70,0.00,1.70,1,212000126,280,15-03-05
2,012A55E5551C95A559A7B074AE85117E,22FA6C39B7A15ABB6EB568E66A6B30E1,6 Inch,6 Inch Sub,B.M.T. 6 inch,12631,0,0.3230,3.64,0.61,4.25,1,121000008,8,15-03-05
3,01B07F1C6DC6945796571FC78735DA16,915B69612845BBD974C39AC105F5CA8C,6 Inch,6 Inch Sub,Veggie Delite 6 inch,12631,0,0.3328,3.75,0.00,3.75,1,121000001,91,15-03-05
4,02DE19974DE5304BF243A1C2EC639495,CD6386C802450EDCF13A3D112D1F86EC,Add On,Add On 6in Portion,Avocado Add6in,12631,0,0.0443,0.50,0.00,0.50,1,175000062,564,15-03-05
5,03F5700F4CB7F5388F217E1532F9A0C3,2B6FD9058A4A28E5E298C95D3C615B27,6 Inch,6 Inch Sub,Turkey 6 inch,46673,0,0.3519,3.91,1.34,5.25,1,121000012,13,15-03-05
6,04420CE325EFCE905A9155CBE6EBC001,72BC117B86F77DF24BEB37C0B191B7B1,Footlong,Footlong Sub,Spicy Italian FtLong,12631,0,0.4881,5.50,0.00,5.50,1,111000002,69,15-03-05
7,0468A23C2C56158FC8F3A0157EE8CEFA,39EEA7FA648C8EAF5CFB4B1EDD18E30F,Bottled Beverage,Bottled Drink,Bottled Carbonated Drink,12631,0,0.1553,1.75,0.00,1.75,1,221000133,279,15-03-05
8,05CDE5B75E2F116B67B1A14699CDA7E5,823D1F989A272E2D215585F695787810,6 Inch,6 Inch Sub,Chicken Teriyaki 6 inch,12631,0,0.3301,3.72,0.78,4.50,1,121000018,34,15-03-05
9,07B2386EF54E80A87955D7B3552D9F3B,902FC16C19F44F8E14F0E1FDAFEB71A3,6 Inch,6 Inch Sub,Ham 6 inch,12631,0,0.3328,3.75,0.00,3.75,1,121000004,2,15-03-05



Типы данных:


MD5KEY_MENUITEM           object
MD5KEY_ORDERSALE          object
CategoryDescription       object
DepartmentDescription     object
Description               object
StoreNumber                int64
TaxInclusiveAmount         int64
TaxAmount                float64
AdjustedPrice            float64
DiscountAmount           float64
Price                    float64
Quantity                   int64
PLU                        int64
Id                         int64
date                      object
dtype: object


Основные статистические показатели:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
MD5KEY_MENUITEM,91431,91431,001E0224E4EDA63D6FED96D59E0C1E76,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MD5KEY_ORDERSALE,91431,40622,2F020569BF698276FA1A93F15330B9AA,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CategoryDescription,91431,15,Footlong,23653,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DepartmentDescription,91431,34,Footlong Sub,22812,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,91431,282,Chips,14154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StoreNumber,91431.0,NaN,NaN,NaN,18152.995319,14823.097615,4904.0,4904.0,12631.0,20974.0,46673.0
TaxInclusiveAmount,91431.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TaxAmount,91431.0,NaN,NaN,NaN,0.263194,0.291208,0.0,0.0949,0.1507,0.4447,43.0312
AdjustedPrice,91431.0,NaN,NaN,NaN,3.464881,3.511239,0.0,1.27,2.1,5.5,506.25
DiscountAmount,91431.0,NaN,NaN,NaN,0.200753,0.672092,-0.75,0.0,0.0,0.13,29.25


In [23]:
numeric_columns = sales.select_dtypes(include=np.number).columns.tolist()

print("Числовые столбцы:", numeric_columns)

Числовые столбцы: ['StoreNumber', 'TaxInclusiveAmount', 'TaxAmount', 'AdjustedPrice', 'DiscountAmount', 'Price', 'Quantity', 'PLU', 'Id']


In [24]:
for col in numeric_columns:
    print(f"\nСтолбец: {col}")
    print("Нулевых значений:", (sales[col] == 0).sum())
    print("Отрицательных значений:", (sales[col] < 0).sum())
    print("Пропущенных значений:", sales[col].isna().sum())


Столбец: StoreNumber
Нулевых значений: 0
Отрицательных значений: 0
Пропущенных значений: 0

Столбец: TaxInclusiveAmount
Нулевых значений: 91431
Отрицательных значений: 0
Пропущенных значений: 0

Столбец: TaxAmount
Нулевых значений: 12140
Отрицательных значений: 0
Пропущенных значений: 0

Столбец: AdjustedPrice
Нулевых значений: 1300
Отрицательных значений: 0
Пропущенных значений: 0

Столбец: DiscountAmount
Нулевых значений: 53243
Отрицательных значений: 10
Пропущенных значений: 0

Столбец: Price
Нулевых значений: 1
Отрицательных значений: 0
Пропущенных значений: 0

Столбец: Quantity
Нулевых значений: 0
Отрицательных значений: 0
Пропущенных значений: 0

Столбец: PLU
Нулевых значений: 0
Отрицательных значений: 0
Пропущенных значений: 0

Столбец: Id
Нулевых значений: 0
Отрицательных значений: 0
Пропущенных значений: 0


In [25]:
date_columns = [
    col for col in sales.columns
    if "date" in col.lower()
]

for col in date_columns:
    dates = pd.to_datetime(sales[col], errors="coerce")

    print(f"\nСтолбец: {col}")
    print("Некорректных дат:", dates.isna().sum())
    print("Минимальная дата:", dates.min())
    print("Максимальная дата:", dates.max())


Столбец: date
Некорректных дат: 0
Минимальная дата: 2001-04-15 00:00:00
Максимальная дата: 2031-05-15 00:00:00


In [35]:
for col in date_columns:
    dates = pd.to_datetime(
        sales[col],
        format="%y-%m-%d",
        errors="coerce"
    )

    outside_period = (
        (dates < "2015-03-01") |
        (dates >= "2015-06-16")
    )

    print(
        f"{col}: записей вне периода —",
        outside_period.sum()
    )

date: записей вне периода — 0


In [36]:
for keyword in [
    "menu_items",
    "recipes",
    "recipes_ingredient_assignments",
    "recipe_sub_recipe_assignments",
    "sub_recipes",
    "sub_recipe_ingr_assignments",
    "ingredients",
    "portion_uom_types"
]:
    matches = [name for name in tables if keyword in name]

    for name in matches:
        print("=" * 80)
        print("ТАБЛИЦА:", name)
        print("Размер:", tables[name].shape)
        print("Столбцы:", tables[name].columns.tolist())
        display(tables[name].head(5))

ТАБЛИЦА: menu_items
Размер: (8005, 5)
Столбцы: ['MenuItemName', 'MenuItemDescription', 'PLU', 'MenuItemId', 'RecipeId']


,MenuItemName,MenuItemDescription,PLU,MenuItemId,RecipeId
0,FtL/Ham,Ham FtLong,111000004,1,6
1,Six/Ham,Ham 6 inch,121000004,2,7
2,FfB/Ham,Ham FtFbd,112000004,3,2
3,fBd/Ham,Ham FlatBd,122000004,4,8
4,Sld/Ham,Ham Salad,131000004,5,9


ТАБЛИЦА: recipes
Размер: (7458, 3)
Столбцы: ['RecipeName', 'RecipeDescription', 'RecipeId']


,RecipeName,RecipeDescription,RecipeId
0,FtL/Ham,Ham FtLong,1
1,FfB/Ham,Ham FtFbd,2
2,Six/Ham,Ham 6 inch,3
3,fBd/Ham,Ham FlatBd,4
4,Sld/Ham,Ham Salad,5


ТАБЛИЦА: sub_recipes
Размер: (81, 3)
Столбцы: ['SubRecipeName', 'SubRecipeDescription', 'SubRecipeId']


,SubRecipeName,SubRecipeDescription,SubRecipeId
0,Std Footlong Sub bread,Std Footlong Sub Bread,1
1,Std 6 Inch Sub bread,Std 6 Inch Sub Bread,2
2,Std Footlong Flatbread Bread,Std Footlong Flatbread Bread,3
3,Std 6 Inch Flatbread bread,Std 6 Inch Flatbread bread,4
4,"Std Salad Lettuce, bowl, & dressing","Std Salad Lettuce, Bowl & Dressing",5


... (6 outputs left out)

ТАБЛИЦА: ingredients
Размер: (673, 4)
Столбцы: ['IngredientName', 'IngredientShortDescription', 'IngredientId', 'PortionUOMTypeId']


,IngredientName,IngredientShortDescription,IngredientId,PortionUOMTypeId
0,Bread ...,Bread Stick 12 inch/30cm,1,11.0
1,Bacon ...,Bacon,2,11.0
2,Chicken Strips ...,Chicken Strips,3,15.0
3,Chicken. oven roasted patty ...,"Chicken, single piece",4,15.0
4,Cold Cut Combo ...,Cold Cut Combo,5,15.0


ТАБЛИЦА: portion_uom_types
Размер: (5, 2)
Столбцы: ['PortionTypeDescription', 'PortionUOMTypeId']


,PortionTypeDescription,PortionUOMTypeId
0,Each,11
1,Fluid Ounce,12
2,Gram,13
3,Milliliter,14
4,Ounce,15


In [37]:
for name, df in tables.items():
    print(f"\n{name}")
    print(df.columns.tolist())


ingredients
['IngredientName', 'IngredientShortDescription', 'IngredientId', 'PortionUOMTypeId']

menuitem
['MD5KEY_MENUITEM', 'MD5KEY_ORDERSALE', 'CategoryDescription', 'DepartmentDescription', 'Description', 'StoreNumber', 'TaxInclusiveAmount', 'TaxAmount', 'AdjustedPrice', 'DiscountAmount', 'Price', 'Quantity', 'PLU', 'Id', 'date']

menu_items
['MenuItemName', 'MenuItemDescription', 'PLU', 'MenuItemId', 'RecipeId']

portion_uom_types
['PortionTypeDescription', 'PortionUOMTypeId']

pos_ordersale
['MD5KEY_ORDERSALE', 'ChangeReceived', 'OrderNumber', 'TaxInclusiveAmount', 'TaxAmount', 'MealLocation', 'TransactionId', 'StoreNumber', 'date']

recipes
['RecipeName', 'RecipeDescription', 'RecipeId']

recipe_ingredient_assignments
['RecipeId', 'IngredientId', 'Quantity']

recipe_sub_recipe_assignments
['RecipeId', 'SubRecipeId', 'Factor']

store_restaurant_sheet1
['STORE_ADDRESS1', 'STORE_ADDRESS2', 'DISTRIBUTION_REGION', 'STORE_STATE', 'STORE_CITY', 'STORE_ZIP', 'STORE_TYPE', 'STORE_LOYAL

In [38]:
def check_relation(
    source_df,
    source_column,
    target_df,
    target_column,
    relation_name
):
    source_values = set(source_df[source_column].dropna())
    target_values = set(target_df[target_column].dropna())

    missing = source_values - target_values

    print(f"\n{relation_name}")
    print("Уникальных ссылок:", len(source_values))
    print("Существующих значений:", len(source_values) - len(missing))
    print("Несуществующих ссылок:", len(missing))

    if missing:
        print("Примеры отсутствующих значений:", list(missing)[:10])

In [40]:
# Проверь названия таблиц и полей по результатам предыдущих ячеек.

check_relation(
    tables["menu_items"],
    "RecipeId",
    tables["recipes"],
    "RecipeId",
    "Связь блюд с рецептами"
)

check_relation(
    tables["recipe_ingredient_assignments"],
    "IngredientId",
    tables["ingredients"],
    "IngredientId",
    "Связь рецептов с ингредиентами"
)

check_relation(
    tables["sub_recipe_ingr_assignments"],
    "IngredientId",
    tables["ingredients"],
    "IngredientId",
    "Связь полуфабрикатов с ингредиентами"
)


Связь блюд с рецептами
Уникальных ссылок: 6922
Существующих значений: 6922
Несуществующих ссылок: 0

Связь рецептов с ингредиентами
Уникальных ссылок: 465
Существующих значений: 465
Несуществующих ссылок: 0

Связь полуфабрикатов с ингредиентами
Уникальных ссылок: 58
Существующих значений: 58
Несуществующих ссылок: 0


In [41]:
quantity_candidates = [
    name for name in tables
    if "ingredient_assignments" in name
]

for name in quantity_candidates:
    df = tables[name]

    quantity_columns = [
        col for col in df.columns
        if "quantity" in col.lower() or "factor" in col.lower()
    ]

    print("\nТАБЛИЦА:", name)

    for col in quantity_columns:
        values = pd.to_numeric(df[col], errors="coerce")

        print(f"\nСтолбец: {col}")
        print("Пропуски:", values.isna().sum())
        print("Нулевые значения:", (values == 0).sum())
        print("Отрицательные значения:", (values < 0).sum())
        print("Положительные значения:", (values > 0).sum())


ТАБЛИЦА: recipe_ingredient_assignments

Столбец: Quantity
Пропуски: 0
Нулевые значения: 542
Отрицательные значения: 0
Положительные значения: 13766


In [42]:
quality_report = []

for name, df in tables.items():
    quality_report.append({
        "Таблица": name,
        "Количество записей": len(df),
        "Количество полей": len(df.columns),
        "Пропущенные значения": int(df.isna().sum().sum()),
        "Полные дубликаты": int(df.duplicated().sum()),
        "Числовые поля": len(df.select_dtypes(include=np.number).columns),
        "Текстовые поля": len(df.select_dtypes(include="object").columns)
    })

quality_report_df = pd.DataFrame(quality_report)

display(
    quality_report_df.sort_values(
        by="Количество записей",
        ascending=False
    )
)

,Таблица,Количество записей,Количество полей,Пропущенные значения,Полные дубликаты,Числовые поля,Текстовые поля
1,menuitem,91431,15,0,0,9,6
4,pos_ordersale,43228,9,0,0,7,2
6,recipe_ingredient_assignments,14308,3,0,0,3,0
7,recipe_sub_recipe_assignments,13890,3,0,0,3,0
2,menu_items,8005,5,0,0,3,2
5,recipes,7458,3,0,0,1,2
0,ingredients,673,4,36,0,2,2
10,sub_recipe_ingr_assignments,379,3,0,0,3,0
9,sub_recipes,81,3,0,0,1,2
3,portion_uom_types,5,2,0,0,1,1


In [43]:
sales = tables["menuitem"].copy()

print("Записи с нулевой базовой ценой:")
display(
    sales[sales["Price"] == 0][
        ["Description", "StoreNumber", "Price",
         "AdjustedPrice", "DiscountAmount", "Quantity", "date"]
    ]
)

print("\nЗаписи с отрицательной скидкой:")
display(
    sales[sales["DiscountAmount"] < 0][
        ["Description", "StoreNumber", "Price",
         "AdjustedPrice", "DiscountAmount", "Quantity", "date"]
    ]
)

print("\nПримеры записей с нулевой итоговой ценой:")
display(
    sales[sales["AdjustedPrice"] == 0][
        ["Description", "StoreNumber", "Price",
         "AdjustedPrice", "DiscountAmount", "Quantity", "date"]
    ].head(15)
)

Записи с нулевой базовой ценой:


,Description,StoreNumber,Price,AdjustedPrice,DiscountAmount,Quantity,date
5878,12oz Coffee,4904,0.0,0.0,0.0,1,15-03-17



Записи с отрицательной скидкой:


,Description,StoreNumber,Price,AdjustedPrice,DiscountAmount,Quantity,date
29001,Rst Chicken Salad,12631,6.00,6.75,-0.75,1,15-04-11
29895,Avocado Add6in,20974,0.75,1.50,-0.75,1,15-04-12
43444,Bottled Water,12631,1.50,1.61,-0.11,1,15-04-26
43798,Chips,12631,1.10,1.14,-0.04,1,15-04-26
57603,Bottled Water,12631,1.50,1.61,-0.11,1,15-05-10
57856,Chips,12631,1.10,1.14,-0.04,1,15-05-10
66558,Chips,12631,1.10,1.14,-0.04,1,15-05-19
69966,Chips,12631,1.10,1.14,-0.04,1,15-05-23
70006,Bottled Water,12631,1.50,1.56,-0.06,1,15-05-23
91164,Chips,12631,1.10,1.14,-0.04,1,15-06-15



Примеры записей с нулевой итоговой ценой:


,Description,StoreNumber,Price,AdjustedPrice,DiscountAmount,Quantity,date
35,Cookie,12631,0.60,0.0,0.60,1,15-03-05
72,Turkey FtLong,12631,6.50,0.0,6.50,1,15-03-05
234,Steak & Chse FtLong,12631,7.25,0.0,7.25,1,15-03-05
322,Steak & Chse 6 inch,12631,4.50,0.0,4.50,1,15-03-05
493,Cookie,12631,0.55,0.0,0.55,1,15-03-06
659,Meatball FtLong,12631,5.50,0.0,5.50,1,15-03-06
862,Chicken Bcn Ranch FtLong,12631,7.25,0.0,7.25,1,15-03-07
1402,Ham FtLong,12631,5.00,0.0,5.00,1,15-03-08
1419,Turkey 6 inch,12631,4.25,0.0,4.25,1,15-03-08
1913,Cookie,12631,0.55,0.0,0.55,1,15-03-10
